In [ ]:
from src.model import SPDMatrixLearner
from src.utils import encode_df, nanmin, nanmax
from src.compute_text_representations import compute_text_representations

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.templates.default = "simple_white"
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
import numpy as np
import torch
from itertools import combinations

torch.set_float32_matmul_precision("medium")
import plotly.express as px
from time import time
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import logging
from sklearn.model_selection import StratifiedKFold

logger = logging.getLogger(__name__)

In [ ]:
df = pd.read_csv("datasets/relative_clause.csv")
sentences = df["sentence"].tolist()
df = df.drop(columns="sentence")

In [ ]:
embeddings = compute_text_representations(
    sentences, model_name="bert-base-uncased", token_aggregation="mean"
)

In [ ]:
device = "cuda"

In [ ]:
X = encode_df(df).to(device)
Y = embeddings[5].to(device)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=0
)

In [ ]:
train_dataset = PairwiseDistanceDataset(X_train, Y_train, distance=2)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=lambda x: x[0],
)
test_dataset = PairwiseDistanceDataset(X_test, Y_test, distance=2)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=lambda x: x[0],
)

In [ ]:
model = SPDMatrixLearner(X.shape[1], param="cholesky")
model = torch.compile(model)
model = model.to(device)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=0.1, maximize=True, weight_decay=1
)
i = 0
prev_w = model.get_W().clone()
model.train()
for (X_batch, Y_batch), (X_batch_test, Y_batch_test) in zip(
    train_dataloader, test_dataloader
):
    t = time()
    start = time()
    optimizer.zero_grad()
    Y_pred = model(X_batch)
    loss = model.loss(Y_pred, Y_batch)
    loss.backward()
    grad_norm = model.compute_gradient_norm()
    optimizer.step()
    rho = model.spearman(Y_pred, Y_batch)
    t_ = time()
    train_duration = t_ - t
    t = t_
    with torch.no_grad():
        model.eval()
        Y_pred = model(X_batch_test)
        test_loss = model.loss(Y_pred, Y_batch_test)
        test_rho = model.spearman(Y_pred, Y_batch_test)
    test_duration = time() - t
    W = model.get_W()
    diff_norm = (W - prev_w).norm(p="fro")
    print(
        f"Epoch {i} - Batch size {len(X_batch):.2g} - Train Loss: {loss.item():.3g} - Train Spearman: {rho:.3g} - Test Loss: {test_loss:.2g} - Test Spearman: {test_rho:.2g} - Train Duration: {train_duration:.2g}s - Gradient Norm: {grad_norm:.2g} - Diff norm {diff_norm:.2g} - Orig param fro {model.W.parametrizations.weight.original.norm(p="fro"):.2g} - Train Min: {train_dataset.min:.2g} - Train Max: {train_dataset.max:.2g}"
    )
    prev_w = model.get_W().clone()
    i += 1
    if grad_norm < 0.01:
        break
    if diff_norm < 0.01:
        break
    if i > 50:
        break
print("")
model.check_spd()
W = pd.DataFrame(
    model.get_W().cpu().detach().numpy(), columns=df.columns, index=df.columns
)
px.imshow(W, color_continuous_midpoint=0, height=800)

In [ ]:
dataset = PairwiseDistanceDataset(
    X, Y, n_pairs=4096, gamma=1.005, distance=2, min_max_scale=False
)
dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=lambda x: x[0],
)

In [ ]:
model = SPDMatrixLearner(X.shape[1], param="exp")
model = torch.compile(model)
model = model.to(device)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=0.1, maximize=True, weight_decay=0
)
i = 0
prev_w = model.get_W().clone()
model.train()
best_score = -torch.inf
patience_counter = 0
for X_batch, Y_batch in dataloader:
    t = time()
    start = time()
    optimizer.zero_grad()
    Y_pred = model(X_batch)
    score = model.loss(Y_pred, Y_batch)
    score.backward()
    grad_norm = model.compute_gradient_norm()
    optimizer.step()
    rho = model.spearman(Y_pred, Y_batch)
    t_ = time()
    train_duration = t_ - t
    W = model.get_W()
    diff_norm = (W - prev_w).norm(p="fro")
    print(
        f"Epoch {i} - Batch size {len(X_batch):.2g} - Train Score: {score.item():.3g} - Train Spearman: {rho:.3g} - Train Duration: {train_duration:.2g}s - Gradient Norm: {grad_norm:.2g} - Diff norm {diff_norm:.2g} - Orig param fro {model.W.parametrizations.weight.original.norm(p="fro"):.2g} - Train Min: {dataset.min:.2g} - Train Max: {dataset.max:.2g}"
    )
    # if rho > best_score:
    #     best_W = model.get_W().clone()
    #     best_score = rho
    #     patience_counter = 0
    # else:
    #     patience_counter += 1
    # if patience_counter > 10:
    #     print("Early stopping")
    #     break
    if diff_norm < 0.01:
        break
    prev_w = model.get_W().clone()
    i += 1
    if i > 200:
        break
print("")
model.check_spd()

In [ ]:
W = model.get_W().cpu().detach()
W += W.triu(1).T
W = torch.where(W.tril() == 0, torch.nan, W)
W = pd.DataFrame(
    W,
    columns=df.columns,
    index=df.columns,
)
px.imshow(W, color_continuous_midpoint=0, height=800)

In [ ]:
px.imshow(
    pd.DataFrame(X_batch.cpu(), columns=df.columns).corr(),
    color_continuous_midpoint=0,
    height=800,
)

In [ ]:
dataset = PairwiseDistanceDataset(
    X, Y, n_pairs=4096 * 2, gamma=1.005, distance=2, min_max_scale=False
)
dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=lambda x: x[0],
)
for X_batch, Y_batch in dataloader:
    break
corrs = (X_batch[:, None] * X_batch[:, :, None])[:, x, y].cpu()
corrs = pd.DataFrame(corrs, columns=feature_pairs).corr().fillna(0)

In [ ]:
from sklearn.cluster import AgglomerativeClustering

agg = AgglomerativeClustering(
    n_clusters=None,
    metric="precomputed",
    distance_threshold=0.5,
    linkage="single",
)
agg.fit(1 - corrs.abs())
s = pd.DataFrame(
    [feature_pairs, agg.labels_], index=["Feature", "Group"]
).T.sort_values("Group")
s["count"] = s.groupby("Group").transform("count")
s[s["count"] > 1].sort_values(["Group", "count"], ascending=[True, False])

In [ ]:
px.imshow(
    pd.DataFrame(X.cpu(), columns=df.columns).corr().fillna(0), height=800
)

In [ ]:
px.imshow(corrs.loc[s.Feature, s.Feature], height=800)

In [ ]:
dataset = PairwiseDistanceDataset(
    X, Y, n_pairs=4096, gamma=1, distance=2, min_max_scale=False
)
dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=lambda x: x[0],
)

In [ ]:
data = []
for i, (X_batch, _) in tqdm(enumerate(dataloader), total=5):
    corrs = (X_batch[:, None] * X_batch[:, :, None])[:, x, y].cpu()
    corrs = pd.DataFrame(corrs, columns=feature_pairs).corr().fillna(0)
    data.append(corrs)
    if i > 4:
        break

In [ ]:
(X_batch[:, None] * X_batch[:, :, None])[:, x, y].shape